In [1]:
import formulallm.formula as f

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker
from llama_index.llms.ollama import Ollama

In [2]:
llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

In [3]:
def load(filePath: str) -> str:
    """Load the Formula DSL code from the file path"""
    code = f.load(filePath)
    return code

load_tool = FunctionTool.from_defaults(fn=load)

In [4]:
import io
from contextlib import redirect_stdout

def solve():
    """Try to solve the partial model based on the domain constraints"""
    f = io.StringIO()
    with redirect_stdout(f):
        f.solve("pm", "1", "Mapping.conforms")
    return f.getvalue()

solve_tool = FunctionTool.from_defaults(fn=solve)

In [5]:
def extract(task_id: str):
    """Extract the solve result to the task with task_id.
    If the partial model is solvable, will return the solution.
    Else if the partial model is unsolvable, will return the least unsatisfied core conditions.
    The task_id is a string representing an integer which is incremented by 1 each time we run the solve command."""
    
    f = io.StringIO()
    with redirect_stdout(f):
        f.extract(task_id, "0", "0")
    return f.getvalue()

extract_result_tool = FunctionTool.from_defaults(fn=extract)

In [6]:
react_agent = ReActAgent.from_tools(tools=[load_tool, solve_tool, extract_result_tool], llm=llama3, verbose=True)

In [7]:
response = react_agent.chat("Can you help me load the file from file path './data/MappingExample.4ml'")

print(str(response))

> Running step 7f0dbef8-13a4-4f77-b87c-6f17a644fa61. Step input: Can you help me load the file from file path './data/MappingExample.4ml'


ConnectError: [Errno 61] Connection refused

In [24]:
response = react_agent.chat('solve the loaded partial model using the solve tool')

print(response)

> Running step 8b8e416d-2a58-44eb-9e48-71527a6100fd. Step input: solve the loaded partial model using the solve tool
Thought: I need to use a tool to help me solve the loaded partial model.
Action: solve
Action Input: {}
Parsing text took: 0
Visiting text took: 0
Started solve task with Id 2.
0.02s.
Observation: None
> Running step f4d547ac-134c-427a-9e58-a021e31a6916. Step input: None
Thought: The solve function did not return any result. Now, I need to extract the solve result using the extract tool.
Action: extract
Action Input: {'task_id': 'some_task_id'}
some_task_id is not task id
0.00s.
Observation: None
> Running step 60792e8e-b63c-436b-b309-f3d0b093deef. Step input: None
Thought: The extract function also did not return any result. It seems that the loaded partial model could not be solved or extracted. I will try to answer the question without using any more tools.

Thought: I cannot answer the question with the provided tools.
Answer: Unfortunately, it is not possible to sol

In [25]:
response = react_agent.chat('extract the result from task 2')

print(response)

> Running step 9e1f9647-e013-4695-ab46-9164be95bac1. Step input: extract the result from task 2
Thought: I need to use a tool to help me answer the question.
Action: extract
Action Input: {'task_id': '2'}
Observation: Error: One or more errors occurred. (Unable to load shared library 'libz3' or one of its dependencies. In order to help diagnose loading problems, consider setting the DYLD_PRINT_LIBRARIES environment variable: 
dlopen(/usr/local/share/dotnet/shared/Microsoft.NETCore.App/8.0.6/libz3.dylib, 0x0001): tried: '/usr/local/share/dotnet/shared/Microsoft.NETCore.App/8.0.6/libz3.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/share/dotnet/shared/Microsoft.NETCore.App/8.0.6/libz3.dylib' (no such file), '/usr/local/share/dotnet/shared/Microsoft.NETCore.App/8.0.6/libz3.dylib' (no such file)
dlopen(/Users/finn/.dotnet/tools/.store/vuisis.formula.arm64/1.0.0/vuisis.formula.arm64/1.0.0/tools/net6.0/any/libz3.dylib, 0x0001): tried: '/Users/finn/.dotnet/tools/.store